In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
# Read Bronze tables
bronze_train = spark.table("airlinepassengers.airlinedata.bronze_train")
bronze_test = spark.table("airlinepassengers.airlinedata.bronze_test")
silver_train = bronze_train
silver_test = bronze_test

In [0]:
silver_train = silver_train.dropDuplicates()
silver_test = silver_test.dropDuplicates()

In [0]:
numeric_columns = [
    "Age",
    "Flight_Distance",
    "Inflight_wifi_service",
    "Departure_Arrival_time_convenient",
    "Ease_of_Online_booking",
    "Gate_location",
    "Food_and_drink",
    "Online_boarding",
    "Seat_comfort",
    "Inflight_entertainment",
    "On_board_service",
    "Leg_room_service",
    "Baggage_handling",
    "Checkin_service",
    "Inflight_service",
    "Cleanliness",
    "Departure_Delay_in_Minutes",
    "Arrival_Delay_in_Minutes"
]

silver_train = silver_train.fillna(0, subset=numeric_columns)
silver_test = silver_test.fillna(0, subset=numeric_columns)

In [0]:
string_columns = [
    "Gender",
    "Customer_Type",
    "Type_of_Travel",
    "Class",
    "satisfaction"
]

silver_train = silver_train.fillna("Unknown", subset=string_columns)
silver_test = silver_test.fillna("Unknown", subset=string_columns)

In [0]:
for c in silver_train.columns:
    silver_train = silver_train.withColumnRenamed(c, c.lower())

for c in silver_test.columns:
    silver_test = silver_test.withColumnRenamed(c, c.lower())

In [0]:
text_columns = [
    "gender",
    "customer_type",
    "type_of_travel",
    "class",
    "satisfaction"
]

for c in text_columns:
    silver_train = silver_train.withColumn(
        c,
        F.lower(F.trim(F.col(c)))
    )
    silver_test = silver_test.withColumn(
        c,
        F.lower(F.trim(F.col(c)))
    )

In [0]:
silver_train = silver_train.filter(
    (F.col("age").cast("double") >= 0) &
    (F.col("age").cast("double") <= 120)
)

silver_test = silver_test.filter(
    (F.col("age").cast("double") >= 0) &
    (F.col("age").cast("double") <= 120)
)
silver_train = silver_train.filter(
    F.col("flight_distance").cast("double") >= 0
)
silver_test = silver_test.filter(
    F.col("flight_distance").cast("double") >= 0
)
silver_train = silver_train.filter(
    (F.col("departure_delay_in_minutes").cast("double") >= 0) &
    (F.col("arrival_delay_in_minutes").cast("double") >= 0)
)
silver_test = silver_test.filter(
    (F.col("departure_delay_in_minutes").cast("double") >= 0) &
    (F.col("arrival_delay_in_minutes").cast("double") >= 0)
)

In [0]:
valid_values = [
    "satisfied",
    "neutral or dissatisfied"
]
silver_train = silver_train.filter(
    F.col("satisfaction").isin(valid_values)
)
silver_test = silver_test.filter(
    F.col("satisfaction").isin(valid_values)
)

In [0]:
print("TRAIN")
silver_train.printSchema()

print("TEST")
silver_test.printSchema()

TRAIN
root
 |-- _c0: string (nullable = true)
 |-- id: string (nullable = true)
 |-- gender: string (nullable = false)
 |-- customer_type: string (nullable = false)
 |-- age: string (nullable = true)
 |-- type_of_travel: string (nullable = false)
 |-- class: string (nullable = false)
 |-- flight_distance: string (nullable = true)
 |-- inflight_wifi_service: string (nullable = true)
 |-- departure_arrival_time_convenient: string (nullable = true)
 |-- ease_of_online_booking: string (nullable = true)
 |-- gate_location: string (nullable = true)
 |-- food_and_drink: string (nullable = true)
 |-- online_boarding: string (nullable = true)
 |-- seat_comfort: string (nullable = true)
 |-- inflight_entertainment: string (nullable = true)
 |-- on_board_service: string (nullable = true)
 |-- leg_room_service: string (nullable = true)
 |-- baggage_handling: string (nullable = true)
 |-- checkin_service: string (nullable = true)
 |-- inflight_service: string (nullable = true)
 |-- cleanliness: str

In [0]:
silver_train.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airlinepassengers.airlinedata.silver_train")

silver_test.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airlinepassengers.airlinedata.silver_test")

In [0]:
from pyspark.sql import functions as F

age_train_col = next(c for c in silver_train.columns if c.lower() == "age")
flight_distance_train_col = next(c for c in silver_train.columns if c.lower() == "flight_distance")
departure_delay_train_col = next(c for c in silver_train.columns if c.lower() == "departure_delay_in_minutes")
arrival_delay_train_col = next(c for c in silver_train.columns if c.lower() == "arrival_delay_in_minutes")

age_test_col = next(c for c in silver_test.columns if c.lower() == "age")
flight_distance_test_col = next(c for c in silver_test.columns if c.lower() == "flight_distance")
departure_delay_test_col = next(c for c in silver_test.columns if c.lower() == "departure_delay_in_minutes")
arrival_delay_test_col = next(c for c in silver_test.columns if c.lower() == "arrival_delay_in_minutes")

invalid_records_train = silver_train.filter(
    (F.col(age_train_col).cast("double") < 0) |
    (F.col(age_train_col).cast("double") > 120) |
    (F.col(flight_distance_train_col).cast("double") < 0) |
    (F.col(departure_delay_train_col).cast("double") < 0) |
    (F.col(arrival_delay_train_col).cast("double") < 0)
)

invalid_records_test = silver_test.filter(
    (F.col(age_test_col).cast("double") < 0) |
    (F.col(age_test_col).cast("double") > 120) |
    (F.col(flight_distance_test_col).cast("double") < 0) |
    (F.col(departure_delay_test_col).cast("double") < 0) |
    (F.col(arrival_delay_test_col).cast("double") < 0)
)

In [0]:
rejected_train = silver_train.filter(
    F.col("id").isNull() |
    F.col("gender").isNull() |
    F.col("customer_type").isNull() |
    F.col("age").isNull() |
    F.col("satisfaction").isNull()
)

rejected_test = silver_test.filter(
    F.col("id").isNull() |
    F.col("gender").isNull() |
    F.col("customer_type").isNull() |
    F.col("age").isNull() |
    F.col("satisfaction").isNull()
)

In [0]:
valid_satisfaction = [
    "satisfied",
    "neutral or dissatisfied"
]
dq_failures_train = silver_train.filter(
    ~F.col("satisfaction").isin(valid_satisfaction)
)
dq_failures_test = silver_test.filter(
    ~F.col("satisfaction").isin(valid_satisfaction)
)

In [0]:
invalid_records_train = silver_train.filter(
    (F.col("age").cast("double") < 0) |
    (F.col("age").cast("double") > 120) |
    (F.col("flight_distance").cast("double") < 0) |
    (F.col("departure_delay_in_minutes").cast("double") < 0) |
    (F.col("arrival_delay_in_minutes").cast("double") < 0)
)

invalid_records_test = silver_test.filter(
    (F.col("age").cast("double") < 0) |
    (F.col("age").cast("double") > 120) |
    (F.col("flight_distance").cast("double") < 0) |
    (F.col("departure_delay_in_minutes").cast("double") < 0) |
    (F.col("arrival_delay_in_minutes").cast("double") < 0)
)

invalid_records_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.invalid_train")

invalid_records_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.invalid_test")

rejected_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.rejected_train")

rejected_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.rejected_test")

dq_failures_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.dq_failures_train")

dq_failures_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.dq_failures_test")

In [0]:
from pyspark.sql import Row

reject_summary = spark.createDataFrame([
    Row(
        invalid_train=invalid_records_train.count(),
        invalid_test=invalid_records_test.count(),
        rejected_train=rejected_train.count(),
        rejected_test=rejected_test.count(),
        dq_failure_train=dq_failures_train.count(),
        dq_failure_test=dq_failures_test.count()
    )
])

display(reject_summary)

invalid_train,invalid_test,rejected_train,rejected_test,dq_failure_train,dq_failure_test
0,0,0,0,0,0


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DoubleType, LongType
)
from datetime import datetime, UTC
import uuid

run_id = str(uuid.uuid4())

job_start_time = datetime.now(UTC)

try:
    # Record counts
    records_received = bronze_train.count() + bronze_test.count()

    records_processed = silver_train.count() + silver_test.count()

    records_rejected = (
        rejected_train.count()
        + rejected_test.count()
        + invalid_records_train.count()
        + invalid_records_test.count()
    )

    validation_failures = (
        dq_failures_train.count()
        + dq_failures_test.count()
    )

    processing_status = "SUCCESS"
    error_message = None

except Exception as e:

    records_received = 0
    records_processed = 0
    records_rejected = 0
    validation_failures = 0

    processing_status = "FAILED"
    error_message = str(e)

job_end_time = datetime.now(UTC)

processing_duration = (
    job_end_time - job_start_time
).total_seconds()

silver_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("job_start_time", TimestampType(), False),
    StructField("job_end_time", TimestampType(), False),
    StructField("processing_duration_seconds", DoubleType(), False),
    StructField("records_received", LongType(), False),
    StructField("records_processed", LongType(), False),
    StructField("records_rejected", LongType(), False),
    StructField("validation_failures", LongType(), False),
    StructField("processing_status", StringType(), False),
    StructField("error_message", StringType(), True)
])

silver_log_df = spark.createDataFrame([

    Row(
        run_id=run_id,
        job_start_time=job_start_time,
        job_end_time=job_end_time,
        processing_duration_seconds=processing_duration,
        records_received=records_received,
        records_processed=records_processed,
        records_rejected=records_rejected,
        validation_failures=validation_failures,
        processing_status=processing_status,
        error_message=error_message
    )

], schema=silver_log_schema)

display(silver_log_df)

run_id,job_start_time,job_end_time,processing_duration_seconds,records_received,records_processed,records_rejected,validation_failures,processing_status,error_message
4c3fc99f-b66f-44b5-9466-4a458a2fe0c8,2026-06-26T16:42:45.045621Z,2026-06-26T16:42:49.393504Z,4.347883,129880,129880,0,0,SUCCESS,null


In [0]:
silver_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("airlinepassengers.airlinedata.silver_execution_log")
display(
    spark.table("airlinepassengers.airlinedata.silver_execution_log")
)

run_id,job_start_time,job_end_time,processing_duration_seconds,records_received,records_processed,records_rejected,validation_failures,processing_status,error_message
bab7c81f-0725-4264-a32f-5ad7fd52a009,2026-06-26T16:38:49.725853Z,2026-06-26T16:38:49.726048Z,1.95E-4,0,0,0,0,FAILED,name 'bronze_train' is not defined
4c3fc99f-b66f-44b5-9466-4a458a2fe0c8,2026-06-26T16:42:45.045621Z,2026-06-26T16:42:49.393504Z,4.347883,129880,129880,0,0,SUCCESS,null


In [0]:
from pyspark.sql import Row
from datetime import datetime, UTC
import uuid

batch_id = str(uuid.uuid4())
processing_timestamp = datetime.now(UTC)

audit_rows = [

    Row(
        batch_id=batch_id,
        source_layer="Bronze",
        processing_timestamp=processing_timestamp,
        table_name="silver_train",
        record_count=silver_train.count(),
        record_status="SUCCESS"
    ),

    Row(
        batch_id=batch_id,
        source_layer="Bronze",
        processing_timestamp=processing_timestamp,
        table_name="silver_test",
        record_count=silver_test.count(),
        record_status="SUCCESS"
    )

]

silver_audit_df = spark.createDataFrame(audit_rows)

display(silver_audit_df)

batch_id,source_layer,processing_timestamp,table_name,record_count,record_status
3f261e25-bd6b-40cd-a014-2d1a9fc2506e,Bronze,2026-06-26T16:44:26.650727Z,silver_train,103904,SUCCESS
3f261e25-bd6b-40cd-a014-2d1a9fc2506e,Bronze,2026-06-26T16:44:26.650727Z,silver_test,25976,SUCCESS


In [0]:
silver_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("airlinepassengers.airlinedata.silver_audit")
display(
    spark.table("airlinepassengers.airlinedata.silver_audit")
)

batch_id,source_layer,processing_timestamp,table_name,record_count,record_status
3f261e25-bd6b-40cd-a014-2d1a9fc2506e,Bronze,2026-06-26T16:44:26.650727Z,silver_train,103904,SUCCESS
3f261e25-bd6b-40cd-a014-2d1a9fc2506e,Bronze,2026-06-26T16:44:26.650727Z,silver_train,103904,SUCCESS
3f261e25-bd6b-40cd-a014-2d1a9fc2506e,Bronze,2026-06-26T16:44:26.650727Z,silver_test,25976,SUCCESS
3f261e25-bd6b-40cd-a014-2d1a9fc2506e,Bronze,2026-06-26T16:44:26.650727Z,silver_test,25976,SUCCESS


In [0]:
from pyspark.sql import Row

validation_report = [
    Row(
        table_name="silver_train",
        total_records=silver_train.count(),
        invalid_records=invalid_records_train.count(),
        rejected_records=rejected_train.count(),
        validation_failures=dq_failures_train.count()
    ),
    Row(
        table_name="silver_test",
        total_records=silver_test.count(),
        invalid_records=invalid_records_test.count(),
        rejected_records=rejected_test.count(),
        validation_failures=dq_failures_test.count()
    )
]

validation_report_df = spark.createDataFrame(validation_report)

display(validation_report_df)

table_name,total_records,invalid_records,rejected_records,validation_failures
silver_train,103904,0,0,0
silver_test,25976,0,0,0


In [0]:
# SILVER LAYER DELIVERABLES
# 1. Save Silver Delta Tables
silver_train.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airlinepassengers.airlinedata.silver_train")

silver_test.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airlinepassengers.airlinedata.silver_test")


# 2. Save Rejected Records
invalid_records_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.invalid_train")

invalid_records_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.invalid_test")

rejected_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.rejected_train")

rejected_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.rejected_test")

dq_failures_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.dq_failures_train")

dq_failures_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("airlinepassengers.airlinedata.dq_failures_test")


# 3. Save Validation Report
validation_report_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("airlinepassengers.airlinedata.silver_validation_report")


# 4. Save Audit Report
silver_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("airlinepassengers.airlinedata.silver_audit_report")


# 5. Save Execution Logs
silver_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("airlinepassengers.airlinedata.silver_execution_log")


print("===================================")
print("Silver Layer Completed Successfully")
print("===================================")

print(f"Silver Train Records      : {silver_train.count()}")
print(f"Silver Test Records       : {silver_test.count()}")
print(f"Invalid Train Records     : {invalid_records_train.count()}")
print(f"Invalid Test Records      : {invalid_records_test.count()}")
print(f"Rejected Train Records    : {rejected_train.count()}")
print(f"Rejected Test Records     : {rejected_test.count()}")
print(f"DQ Failure Train Records  : {dq_failures_train.count()}")
print(f"DQ Failure Test Records   : {dq_failures_test.count()}")

Silver Layer Completed Successfully
Silver Train Records      : 103904
Silver Test Records       : 25976
Invalid Train Records     : 0
Invalid Test Records      : 0
Rejected Train Records    : 0
Rejected Test Records     : 0
DQ Failure Train Records  : 0
DQ Failure Test Records   : 0
